In [ ]:
import agama, numpy
from astropy import units as u 
from astropy import constants as cnst
import pathlib
import configparser

In [2]:
def load_potential_parameters(*ini_paths):
	parameters = {}
	for ini_path in ini_paths:
		config = configparser.ConfigParser()
		config.optionxform = str
		config.read(ini_path)
		for section in config.sections():
			if not section.startswith("Potential "):
				continue
			name = section.removeprefix("Potential ")
			parameters[name] = {
				key: value if key == "type" else float(value)
				for key, value in config[section].items()
			}
	return parameters

def rescale_ini_dict(parameters, unitM, unitL, unitM_OG=u.Msun, unitL_OG=u.kpc):
    """
    Expects the ini files to have been parsed with load_potential_parameters
    """
    parameters_rescaled = {}
    for key,value in parameters.items():
        parameters_rescaled[key] = {}
        for key_,value_ in value.items():
            if key_.startswith("mass"):
                parameters_rescaled[key][key_] = (value_*unitM_OG).to(unitM).value
            elif key_.startswith("scale"):
                parameters_rescaled[key][key_] = (value_*unitL_OG).to(unitL).value
            else:
                parameters_rescaled[key][key_] = value_    
    return parameters_rescaled

In [3]:
path_bar = "/Users/ferrone/repos/bar-dynamics/parameters/components/bar_ferrers_v1.ini"
path_ext = "/Users/ferrone/repos/bar-dynamics/parameters/components/mw_fiducial_v1.ini"
parameters_ext = load_potential_parameters(path_ext)
parameters_bar = load_potential_parameters(path_bar)
barconfig = configparser.ConfigParser()
barconfig.read(path_bar)

['/Users/ferrone/repos/bar-dynamics/parameters/components/bar_ferrers_v1.ini']

set the units 

In [ ]:
oldUnitG    =   (u.km/u.s)**2 * (u.kpc/u.Msun)
G           =   cnst.G.to(oldUnitG)
unitM   =   parameters_bar["bar"]["mass"]*u.Msun
unitL   =   parameters_bar["bar"]["scaleRadius"]*u.kpc
unitV   =   (G*unitM/unitL)**(1/2)
unitRot =   unitV/unitL
agama.setUnits(length=unitL.value, mass=unitM.value, velocity=unitV.value) 

rescale the parameters

In [5]:
parameters_ext_rescaled = rescale_ini_dict(parameters_ext,unitM=unitM,unitL=unitL)
parameters_bar_rescaled = rescale_ini_dict(parameters_bar,unitM=unitM,unitL=unitL)
omega = numpy.float64(barconfig["orientation"]["pattern_speed_kms_kpc"]) * (u.km/u.s)/u.kpc
omega = omega.to(unitRot).value

create the model in agama

In [6]:
thin_disk   = agama.Potential(**parameters_ext['thin_disk'])
thick_disk  = agama.Potential(**parameters_ext['thick_disk'])
halo        = agama.Potential(**parameters_ext['halo'])
bar         = agama.Potential(**parameters_bar["bar"])
potential_external = thin_disk + thick_disk + halo
potential_total = bar + potential_external

sample the initial conditions 

In [7]:
norbits     =   int(1e4)
ics,mass    =   bar.sample(norbits, potential=potential_total, beta=1/2,kappa=1/10)

start getting the density targets 

In [8]:
stripsPerPane   =   2
gridr           =   agama.nonuniformGrid(11, 0.1, 1,)
target_density  =   agama.Target(type = "DensityClassicLinear",
                                 gridr = gridr, 
                                 stripsPerPane = stripsPerPane, 
                                 axisRatioY=parameters_bar["bar"]['axisRatioY'], 
                                 axisRatioZ=parameters_bar["bar"]['axisRatioZ'])
times           =   50*potential_total.Tcirc(ics)

create the orbital library 

In [9]:
D,traj=agama.orbit(ic=ics,
                   potential=potential_total,
                   Omega=omega,
                   dtype=object,
                   time=times,
                   separateTime=True, 
                   targets=target_density)

10000 orbits complete (1818 orbits/s)


set up the schwartzchild model

In [10]:
total_bar_mass = bar.totalMass()
w = numpy.ones(norbits) * total_bar_mass / norbits
mass_discrete   = D.T.dot(w)
mass_true       = target_density(bar) # the right hand side of the inversion 

In [11]:
data0   = numpy.ones((1, norbits))
sigma   = numpy.maximum(0.1*mass_true, 0.05*mass_true.mean())
regul   = 1.0 
weights = agama.solveOpt(
    matrix  = [data0, D.T],
    rhs     = [[total_bar_mass], mass_true],
    rpenq   = [[numpy.inf], 2*sigma**-2],
    xpenq   = regul * 2*numpy.ones(norbits)*norbits/total_bar_mass**2 
)

Using a custom optimized KKT solver
numVariables=10191, numConstraints=192, numConsIneq=10000
Benchmarking BLAS backends for matrix size=(192,10191): cvxopt=0.001 s, numpy=0.002 s, scipy=0.003 s => choosing cvxopt
     pcost       dcost       gap    pres   dres
 0:  9.6377e+02 -1.0000e+10  1e+10  3e-14  1e+08
 1:  9.6377e+02 -9.9719e+07  1e+08  2e-08  1e+06
 2:  9.6333e+02 -1.0102e+06  1e+06  2e-08  1e+04
 3:  9.2266e+02 -2.1395e+04  2e+04  2e-08  2e+02
 4:  4.9961e+02 -8.7199e+03  9e+03  2e-08  7e+01
 5:  3.2657e+02 -7.9350e+03  8e+03  2e-08  4e+01
 6:  1.2981e+02 -1.8760e+03  2e+03  2e-08  8e+00
 7:  2.5244e+01 -7.2681e+02  8e+02  2e-08  2e-15
 8:  5.3641e+00 -1.7657e+02  2e+02  2e-08  3e-16
 9:  2.0805e+00 -4.3958e+01  5e+01  2e-08  1e-17
10:  1.6502e+00 -9.5405e+00  1e+01  3e-08  2e-18
11:  1.5451e+00 -1.9791e+00  4e+00  2e-08  6e-23
12:  1.5104e+00  6.8384e-01  8e-01  2e-08  2e-23
13:  1.4948e+00  1.2611e+00  2e-01  2e-08  1e-23
14:  1.4908e+00  1.4336e+00  6e-02  2e-08  7e-24
15:

sample the initial conditions 

In [12]:
Nbody = 100000
snap_posvel, snap_mass = agama.sampleOrbitLibrary(Nbody, traj, weights)

print the virial ratio as a check

In [13]:
# approximate the potential with a multipole expansion 
pot_stars_nbody =   agama.Potential(type='multipole', particles=(snap_posvel, snap_mass), symmetry='t')
E_kin       =   0.5 * numpy.sum(snap_mass[:,None] * snap_posvel[:,3:6]**2)
W_self      =   0.5*numpy.sum(snap_mass * pot_stars_nbody.potential(snap_posvel[:,0:3]))
force_rest  =   potential_external.force(snap_posvel[:,0:3])                      # -∇Φ_rest, shape (N,3)
W_ext       =   numpy.sum(snap_mass * numpy.sum(snap_posvel[:,0:3]*force_rest, axis=1))
virial_ratio = -2*E_kin / (W_self + W_ext)
print(virial_ratio)

0.9983551724468628
